# 双目手部管线（一条命令）· Colab

给新手：这个笔记本把 HOT3D 公开双目片段当成“假设备”，跑完整条流程：

```mermaid
flowchart LR
  A[HOT3D 片段] --> B[会话目录] --> C[校验] --> D[WiLoR 左右目] --> E[三角化+一致性] --> F[平滑] --> G[世界系 episode] --> H[QC] --> I[LeRobot]
```

需要：GPU 运行时；MANO 文件（自己去 https://mano.is.tue.mpg.de 注册下载，放到 `/content/mano`）；WiLoR 权重。说明见 `docs/stereo_pipeline.md`。

In [ ]:
!git clone -q https://github.com/skyxiaobai/robot.git /content/robot
!git clone -q https://github.com/rolpotamias/WiLoR.git /content/WiLoR
%cd /content/robot
!pip -q install ultralytics==8.1.34 smplx git+https://github.com/facebookresearch/hand_tracking_toolkit
!pip -q install --no-build-isolation chumpy
!pip -q install -r /content/WiLoR/requirements.txt

In [ ]:
# WiLoR 权重（CC-BY-NC-ND，不进仓库），下载地址见 WiLoR README
!mkdir -p /content/WiLoR/pretrained_models
!wget -q -nc https://huggingface.co/spaces/rolpotamias/WiLoR/resolve/main/pretrained_models/detector.pt -P /content/WiLoR/pretrained_models/
!wget -q -nc https://huggingface.co/spaces/rolpotamias/WiLoR/resolve/main/pretrained_models/wilor_final.ckpt -P /content/WiLoR/pretrained_models/
!wget -q -nc https://huggingface.co/spaces/rolpotamias/WiLoR/resolve/main/pretrained_models/model_config.yaml -P /content/WiLoR/pretrained_models/
import os
os.environ.update({
  'MANO_MODEL_DIR': '/content/mano',
  'WILOR_CHECKPOINT': '/content/WiLoR/pretrained_models/wilor_final.ckpt',
  'WILOR_CONFIG': '/content/WiLoR/pretrained_models/model_config.yaml',
  'WILOR_DETECTOR': '/content/WiLoR/pretrained_models/detector.pt',
  'PYTHONPATH': '/content/WiLoR',
})

## 下载几段 HOT3D-Clips（Quest3）

HOT3D 在 HuggingFace `bop-benchmark/hot3d`，需要先在网页上同意许可。把 `train_quest3` 的若干 tar 解压到 `/content/hot3d/train_quest3/clip-XXXXXX/`。

In [ ]:
# 1) 假录制器：生成设备格式的会话目录，并校验（故障注入可以看到校验怎么报错）
!python scripts/simulate_device_session.py --hot3d /content/hot3d/train_quest3/clip-000000 --out /content/sim/sess01 --synth-imu --max-frames 30
!python scripts/validate_session.py /content/sim/sess01
!python scripts/simulate_device_session.py --hot3d /content/hot3d/train_quest3/clip-000000 --out /content/sim/bad --max-frames 30 --fault desync_5ms
!python scripts/validate_session.py /content/sim/bad || echo '↑ 预期不通过'

In [ ]:
# 2) 一条命令跑完：WiLoR → 三角化 → 一致性 → 平滑 → 世界系 → QC → LeRobot
!python scripts/run_stereo_pipeline.py --hot3d /content/hot3d/train_quest3/clip-000000 --out /content/out/hot3d
# 设备到货后换成：python scripts/run_stereo_pipeline.py --session /path/sess01 --out /content/out/sess01

In [ ]:
from IPython.display import Markdown
Markdown(open('/content/out/hot3d/report.md').read())